In [ ]:
# Day 5 processing: QC, clustering and marker analysis of the timecourse.
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
import seaborn as sns
import torch
sc.set_figure_params(figsize=(3, 3),  facecolor="white",
    frameon=True,   dpi = 300, format = 'png')
sns.set_theme()
torch.set_float32_matmul_precision("high")

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [ ]:
# Load data
adata = sc.read_h5ad("/Users/yanjun/Desktop/scanpy/3_1/3_1_day5/3_1_day5.h5ad")

In [ ]:
sc.settings.figdir = "/Users/yanjun/Desktop/PQE_thesis/scripts/chapter3/day5/"

In [ ]:
adata

In [ ]:

cell_counts = adata.obs["condition"].value_counts()
cell_counts

In [ ]:
import seaborn as sns
import scanpy as sc
import matplotlib.pyplot as plt

p1 = sns.displot(adata.obs["total_counts"], bins=100, kde=False)
p1.savefig("day5/p1_total_counts.png", dpi=150, bbox_inches="tight")
 
p2 = sns.displot(adata.obs["log1p_total_counts"], bins=100, kde=False)
p2.savefig("day5/p2_log1p_total_counts.png", dpi=150, bbox_inches="tight")
 
ax3 = sc.pl.violin(adata, "pct_counts_mt", show=False)
ax3.figure.savefig("day5/p3_pct_counts_mt.png", dpi=150, bbox_inches="tight")
 
ax4 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts",
                    color="pct_counts_mt", show=False)
ax4.figure.savefig("day5/p4_counts_vs_genes.png", dpi=150, bbox_inches="tight")
 
plt.close("all")

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True# save = 'qc.png'
)

In [ ]:
s_genes = [
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1", "UNG", "GINS2", 
    "MCM6", "CDCA7", "DTL", "PRIM1", "UHRF1", "MLF1IP", "HELLS", "RFC2", 
    "RPA2", "NASP", "RAD51AP1", "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7", 
    "POLD3", "MSH2", "ATAD2", "RAD51", "RRM2", "CDC45", "CDC6", "EXO1", 
    "TIPIN", "DSCC1", "BLM", "CASP8AP2", "USP1", "CLSPN", "POLA1", "CHAF1B", 
    "BRIP1", "E2F8"
]
g2m_genes = [
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A", "NDC80", 
    "CKS2", "NUF2", "CKS1B", "MKI67", "TMPO", "CENPF", "TACC3", "FAM64A", 
    "SMC4", "CCNB2", "CKAP2L", "CKAP2", "AURKB", "BUB1", "KIF11", "ANP32E", 
    "TUBB4B", "GTSE1", "KIF20B", "HJURP", "CDCA3", "HN1", "CDC20", "TTK", 
    "CDC25C", "KIF2C", "RANGAP1", "NCAPD2", "DLGAP5", "CDCA2", "CDCA8", 
    "ECT2", "KIF23", "HMMR", "AURKA", "PSRC1", "ANLN", "LBR", "CKAP5", 
    "CENPE", "CTCF", "NEK2", "G2E3", "GAS2L3", "CBX5", "CENPA"
]


In [ ]:
# Gene scoring
sc.tl.score_genes_cell_cycle(adata, s_genes, g2m_genes, copy=False)

In [ ]:
adata

In [ ]:
# Normalise, HVG and PCA
adata.X = adata.layers['counts'].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers['data'] = adata.X
sc.pp.highly_variable_genes(adata, n_top_genes = 5000 ,min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pp.scale(adata, max_value=10)
sc.pp.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
sc.pl.pca(adata, color = ["pct_counts_mt",  "S_score", "G2M_score", 
                         "phase"], ncols = 2, save = 'pca.png')

In [ ]:
# Neighbours, UMAP and clustering
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=20)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color = ["pct_counts_mt",  "S_score", "G2M_score", 

                         "phase"], ncols = 2, save = 'umap.png')

In [ ]:
import plotly.express as px
import numpy as np

adata.obs['LYVE1'] = adata[:, 'LYVE1'].X.toarray().flatten()
adata.obs['LYVE1_clipped'] = np.clip(adata.obs['LYVE1'], 0, 3)  # adjust upper limit as needed

fig = px.scatter_3d(
    adata.obs,
    x='n_genes_by_counts',
    y='pct_counts_mt',
    z='total_counts',
    color='LYVE1_clipped',
    opacity=0.5,
    width=800,
    height=600,
    color_continuous_scale='RdBu_r',
    range_color=[0, 3],
)
fig.show()

In [ ]:
adata = adata[adata.obs['pct_counts_mt'] < 50].copy()

In [ ]:

cell_counts = adata.obs["condition"].value_counts()
cell_counts

In [ ]:
sc.pl.pca(adata, color = ["condition", "pct_counts_mt",  "S_score", "G2M_score", 
                         "phase"], ncols = 2, save = 'pca.png')

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50)
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(
    adata,
    resolution= 1,
    random_state=0,
    flavor="igraph",
    n_iterations=2,
    directed=False,
    key_added='leiden_res_1'
)

In [ ]:
sc.tl.leiden(
    adata,
    resolution= 0.6,
    random_state=0,
    flavor="igraph",
    n_iterations=2,
    directed=False,
    key_added='leiden_res_0_6'
)

In [ ]:
sc.pl.umap(adata, color=["leiden_res_0_6","leiden_res_1",  "condition", "S_score", "G2M_score", 
                         "phase", "pct_counts_mt"], legend_loc = 'on data', ncols = 2,
            save = 'umap.png'
          )

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    groupby = "leiden_res_0_6", save = 'multiqc.png'
)

In [ ]:
adata = sc.read_h5ad("/Users/yanjun/Desktop/scanpy/4/day5_50.h5ad")

In [ ]:
# Differential expression
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0_6", method="wilcoxon", layer = 'data')
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="leiden_res_0_6", standard_scale="var", n_genes=20, dendrogram = None#, save = 'res06_top20genes.png'
)

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key[:1]}": result[key][group]
     for group in groups for key in ['names', 'scores','pvals_adj', 'logfoldchanges']}
)
df.to_csv('/Users/yanjun/Desktop/scanpy/3_2/3_2_day5/50/leiden06_deg.csv', index=False)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="condition", method="wilcoxon", layer = 'data')
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="condition", standard_scale="var", n_genes=50, dendrogram = None#, save = 'condition_topgenes.png'
)

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key[:1]}": result[key][group]
     for group in groups for key in ['names', 'scores',  'pvals_adj', 'logfoldchanges']}
)
df.to_csv('/Users/yanjun/Desktop/scanpy/3_2/3_2_day5/50/condition_deg.csv', index=False)

In [ ]:
adata.uns['rank_genes_groups']

In [ ]:
sc.pl.dotplot(adata, ['C1QA','C1QC','CD68','PTPRC','S100A8','LYVE1', 'FOLR2', 'F13A1', 'SELENOP', 'VCAN',
                      'LYN', 'CD14', 'FCGR3A',
                    'MRC1', 'HLA-DRA', 'CD68', 'HLA-A', 'PSAP', 'S100A9'], 
              groupby = 'leiden_res_0_6', dendrogram = None, save = 'keymarkers.pdf'
             )

In [ ]:
# PAGA and diffusion maps
sc.tl.paga(adata, 'leiden_res_0_6')
scv.pl.paga(adata, basis='umap', threshold=0.4, max_edge_width=0.3, alpha=.4, title='',  
            figsize=(4.7,5), legend_loc='right margin')

In [ ]:
sc.pl.scatter(adata, x = 'pct_counts_mt', y = 'LYVE1', color = 'leiden_res_0_6')

In [ ]:
sc.pl.scatter(adata, x = 'pct_counts_mt', y = 'F13A1', color = 'leiden_res_0_6')

In [ ]:
sc.pl.scatter(adata, x = 'pct_counts_mt', y = 'MRC1', color = 'leiden_res_0_6')

In [ ]:
sc.tl.diffmap(adata, n_comps = 30)
scv.pl.diffmap(adata, color='leiden_res_0_6', legend_loc='right margin', figsize=(10,10), 
               size=20, alpha=0.8, title= 'Diffusion Map')

In [ ]:
apop = ['AIFM1', 'AKT1', 'AKT2', 'AKT3', 'APAF1', 'ATM', 'BAD', 'BAX', 'BCL2', 'BCL2L1', 'BID', 'BIRC2', 'BIRC3', 'CAPN1', 'CAPN2', 
        'CASP10', 'CASP3', 'CASP6', 'CASP7', 'CASP8', 'CASP9', 'CFLAR', 'CHUK', 'CSF2RB', 'CYCS', 'DFFA', 'DFFB', 'ENDOG', 'FADD', 
        'FAS', 'FASLG', 'IKBKB', 'IKBKG', 'IL1A', 'IL1B', 'IL1R1', 'IL1RAP', 'IL3', 'IL3RA', 'IRAK1', 'IRAK2', 'IRAK3', 'IRAK4', 'MAP3K14', 
        'MYD88', 'NFKB1', 'NFKB2', 'NFKBIA', 'NTRK1', 'PIK3CA', 'PIK3CB', 'PIK3CD', 'PIK3CG', 'PIK3R1', 'PIK3R2', 'PIK3R3', 'PIK3R5', 
        'PPP3CA', 'PPP3CB', 'PPP3CC', 'PPP3R1', 'PPP3R2', 'PRKACA', 'PRKACB', 'PRKACG', 'PRKAR1A', 'PRKAR1B', 'PRKAR2A', 'PRKAR2B', 'RELA', 
        'RIPK1', 'TNF', 'TNFRSF10A', 'TNFRSF10B', 'TNFRSF10C', 'TNFRSF10D', 'TNFRSF1A', 'TNFSF10', 'TP53', 'TRADD', 'TRAF2']

In [ ]:
sc.tl.score_genes(adata, apop, score_name = 'apoptotic')

In [ ]:
sc.pl.scatter(adata, "pct_counts_mt", "apoptotic", color="leiden_res_0_6")

In [ ]:
adata.obs["pre_annotation"] = adata.obs["leiden_res_0_6"].map(
    {
        "0": "LYVE1hi_RPLhi",
        "1": "LYVE1hi_RPLhi",
        "2": "LYVE1lo_APOEhi",
        "3": "LYVE1lo_MThi",
        "4": "LYVE1lo_unknown",
        "5": "LYVE1hi_MThi",
        "6": "LYVE1hi_ELMOhi",
        "7": "LYVE1lo_ICAMhi"
    }
)

In [ ]:
adata = sc.read_h5ad('/Users/yanjun/Desktop/scanpy/4/day5_50.h5ad')

In [ ]:
adata.obs["toy"] = adata.obs["leiden_res_0_6"].map(
    {
        "0": "Day5_LYVE1hi",
        "1": "Day5_LYVE1hi",
        "2": "Day5_LYVE1lo",
        "3": "Day5_LYVE1lo",
        "4": "Day5_LYVE1lo",
        "5": "Day5_LYVE1hi",
        "6": "Day5_LYVE1hi"
    }
)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby = 'toy', method='wilcoxon', layer = 'data')

In [ ]:
sc.pl.umap(adata, color=["leiden_res_0_6","LYVE1",  "MRC1", "F13A1", 'ICAM1'], legend_loc = 'on data', ncols = 2, size = 100
          )

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, groupby='toy', n_genes=50, use_raw=False, layer='data')

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv('/Users/yanjun/Desktop/scanpy/4/day5_toy.csv', index=False)

In [ ]:
print(df.head(10))

In [ ]:
# Save result
adata.write('/Users/yanjun/Desktop/scanpy/4/day5_50.h5ad')